In [1]:
import json, re
import pandas as pd
 
CUSTOM_FIELD_MAP = {
    "neci8m1ih2": "schedule",
    "6hqa392kmt": "closures_reschedules",
    "r9lac8vy2c": "food_provided",
    "cooifgou2h": "eligibility",
    "t79gvzzldl": "service_notes",
}
 
def load_from_file(path):
    text = open(path, encoding="utf-8").read()
    match = re.search(r"```json\s*\n(.*?)\n```", text, re.DOTALL)
    if not match:
        raise ValueError(f"No json code block found in {path}")
    return json.loads(match.group(1))
 
def parse_custom_fields(raw):
    if not raw:
        return {}
    try:
        parsed = json.loads(raw)
    except (json.JSONDecodeError, TypeError):
        return {}
    return {CUSTOM_FIELD_MAP.get(k, k): v for k, v in parsed.items()}
 
def to_dataframe(payload):
    locations = payload["results"]["locations"]
    rows = []
    for loc in locations:
        custom = parse_custom_fields(loc.get("custom_fields"))
        rows.append({
            "id": loc.get("id"),
            "name": loc.get("name"),
            "address": loc.get("streetaddress"),
            "latitude": loc.get("loc_lat"),
            "longitude": loc.get("loc_long"),
            "phone": loc.get("phone") or None,
            "website": loc.get("website") or None,
            "tags": loc.get("tags"),
            "schedule": custom.get("schedule"),
            "closures_reschedules": custom.get("closures_reschedules"),
            "food_provided": custom.get("food_provided"),
            "eligibility": custom.get("eligibility"),
            "service_notes": custom.get("service_notes"),
        })
    df = pd.DataFrame(rows)
    df["tags_list"] = df["tags"].fillna("").apply(
        lambda s: [t.strip() for t in s.split(",") if t.strip()]
    )
    return df
 
# ---- point this at wherever you saved the file ----
payload = load_from_file("san-diego-food-bank-locations.md")
df = to_dataframe(payload)
 
print(df.shape)
df.head()

(156, 14)


,id,name,address,latitude,longitude,phone,website,tags,schedule,closures_reschedules,food_provided,eligibility,service_notes,tags_list
0,54637170,Aguilas del Poderoso Dios,"5901 Rancho Hills Drive, San Diego, CA 92139",32.672252,-117.061430,NaN,None,"thursday,neighborhood distribution",1st and 3rd Thursday of the month 10:00am unti...,"Reschedules: January 8 and January 22, 2026. C...",25-30 pounds of fresh produce; occasional dry ...,All are welcome,"Walk-up & drive-thru services; if walking up, ...","[thursday, neighborhood distribution]"
1,49902111,All Saint Episcopal Church,"651 Eucalyptus Avenue, Vista, CA 92084",33.202163,-117.234913,NaN,None,"food to nonprofit,saturday",3rd and 4th Saturday of each month; 11 AM - 12...,NaN,Nonperishable dry goods and fresh produce,All are welcome,NaN,"[food to nonprofit, saturday]"
2,49902112,Apostolic Assembly Church,"1717 East Lincoln Avenue, Escondido, CA 92027",33.143996,-117.063386,NaN,None,"saturday,neighborhood distribution",3rd Saturday of the month from 9:30 am to 10:3...,"January 10, 2026",25-30 pounds of fresh produce; occasional dry ...,NaN,Drive-thru services only; Food Bank ID cards a...,"[saturday, neighborhood distribution]"
3,49902113,Apostolic Assembly First Church San Diego,"611 South 35th St, San Diego, CA 92113",32.699690,-117.118156,NaN,None,"thursday,efap",2nd Thursday of each month from 10:00 am - 12:...,No holiday reschedules for 2026,Food menu changes monthly. Typical food packag...,Emergency Food Assistance Program Income Guide...,"Walk-up & drive-thru services; if walking up, ...","[thursday, efap]"
4,49902166,Armed Services YMCA Camp Pendleton,Building 200090 Ash Road and Wire Mountain Roa...,33.222690,-117.380070,NaN,None,"friday,efap",4th Friday of the month from 9:15 am - 11:45 am,"Rescheduled to: May 29, Nov 20, Dec 11",Food menu changes monthly. Typical food packag...,Emergency Food Assistance Program Income Guide...,Military base access required to reach this site.,"[friday, efap]"


In [2]:
df = df.drop(columns=["id"])

In [3]:
import re

SD_COUNTY_CITIES = sorted([
    "Jacumba Hot Springs", "Rancho Santa Fe", "Valley Center", "Borrego Springs",
    "Imperial Beach", "National City", "Lemon Grove", "Spring Valley",
    "Pine Valley", "Chula Vista", "San Marcos", "El Cajon", "La Mesa",
    "San Diego", "Escondido", "Oceanside", "Carlsbad", "Encinitas",
    "Solana Beach", "Del Mar", "Coronado", "Lakeside", "Alpine", "Ramona",
    "Fallbrook", "Bonsall", "Jamul", "Jacumba", "Boulevard", "Descanso",
    "Julian", "Potrero", "Campo", "Tecate", "Vista", "Poway", "Santee",
], key=len, reverse=True)

In [4]:
CITY_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(c) for c in SD_COUNTY_CITIES) + r")\b",
    re.IGNORECASE,
)
FALLBACK_CITY_PATTERN = re.compile(r"([A-Za-z][A-Za-z\s\.]*?),\s*CA\b")
 
SD_COUNTY_PLACES = SD_COUNTY_CITIES + [
    "San Ysidro", "Nestor", "Dulzura", "Pauma Valley", "Guatay",
    "Warner Springs", "Otay Mesa", "Bonita", "Rancho Bernardo",
    "Rancho Penasquitos", "Mira Mesa", "Clairemont", "Pacific Beach",
    "La Jolla", "Point Loma", "Pala", "Pauma", "Mount Laguna",
]
IMPERIAL_COUNTY_PLACES = [
    "El Centro", "Calexico", "Brawley", "Imperial", "Holtville",
    "Westmorland", "Calipatria", "Niland", "Seeley", "Heber",
    "Ocotillo", "Winterhaven",
]
COUNTY_MAP = {}
COUNTY_MAP.update({c.title(): "San Diego County" for c in SD_COUNTY_PLACES})
COUNTY_MAP.update({c.title(): "Imperial County" for c in IMPERIAL_COUNTY_PLACES})

In [5]:
def extract_zip(address):
    if not isinstance(address, str):
        return None
    match = re.search(r"(\d{5})(?:-\d{4})?\s*(?:,?\s*US)?\s*$", address)
    return match.group(1) if match else None

In [6]:
def extract_city(address):
    if not isinstance(address, str):
        return None
    matches = CITY_PATTERN.findall(address)
    if matches:
        return matches[-1].title()
    fallback = FALLBACK_CITY_PATTERN.findall(address)
    return fallback[-1].strip().title() if fallback else None

In [7]:
def get_county(city):
    if not isinstance(city, str):
        return None
    return COUNTY_MAP.get(city.title())

In [8]:
df

,name,address,latitude,longitude,phone,website,tags,schedule,closures_reschedules,food_provided,eligibility,service_notes,tags_list
0,Aguilas del Poderoso Dios,"5901 Rancho Hills Drive, San Diego, CA 92139",32.672252,-117.061430,NaN,None,"thursday,neighborhood distribution",1st and 3rd Thursday of the month 10:00am unti...,"Reschedules: January 8 and January 22, 2026. C...",25-30 pounds of fresh produce; occasional dry ...,All are welcome,"Walk-up & drive-thru services; if walking up, ...","[thursday, neighborhood distribution]"
1,All Saint Episcopal Church,"651 Eucalyptus Avenue, Vista, CA 92084",33.202163,-117.234913,NaN,None,"food to nonprofit,saturday",3rd and 4th Saturday of each month; 11 AM - 12...,NaN,Nonperishable dry goods and fresh produce,All are welcome,NaN,"[food to nonprofit, saturday]"
2,Apostolic Assembly Church,"1717 East Lincoln Avenue, Escondido, CA 92027",33.143996,-117.063386,NaN,None,"saturday,neighborhood distribution",3rd Saturday of the month from 9:30 am to 10:3...,"January 10, 2026",25-30 pounds of fresh produce; occasional dry ...,NaN,Drive-thru services only; Food Bank ID cards a...,"[saturday, neighborhood distribution]"
3,Apostolic Assembly First Church San Diego,"611 South 35th St, San Diego, CA 92113",32.699690,-117.118156,NaN,None,"thursday,efap",2nd Thursday of each month from 10:00 am - 12:...,No holiday reschedules for 2026,Food menu changes monthly. Typical food packag...,Emergency Food Assistance Program Income Guide...,"Walk-up & drive-thru services; if walking up, ...","[thursday, efap]"
4,Armed Services YMCA Camp Pendleton,Building 200090 Ash Road and Wire Mountain Roa...,33.222690,-117.380070,NaN,None,"friday,efap",4th Friday of the month from 9:15 am - 11:45 am,"Rescheduled to: May 29, Nov 20, Dec 11",Food menu changes monthly. Typical food packag...,Emergency Food Assistance Program Income Guide...,Military base access required to reach this site.,"[friday, efap]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
151,Us 4 Warriors,"900 Paseo del Rey, Chula Vista, CA 91910",32.633457,-117.027515,NaN,None,"tuesday,neighborhood distribution",4th Tuesday of the month from 12:00pm - 1:00pm,All distributions will be held on regularly sc...,25-30 pounds of fresh produce; occasional dry ...,All are welcome,"Walk-up & drive-thru services; if walking up, ...","[tuesday, neighborhood distribution]"
152,Victory Resource Center,"1220 Third Avenue, Suite 1 Chula Vista, CA 91911",32.609358,-117.067768,NaN,None,"efap,monday,wednesday,friday","Starting January 28th; Mondays, Wednesdays, an...","Closed in 2026 on: Feb 13, Feb 16, Mar 30, Apr...",Food menu changes monthly. Typical food packag...,Emergency Food Assistance Program Income Guide...,NaN,"[efap, monday, wednesday, friday]"
153,Warner Springs Community Resource Center,"30950 Highway 79, Warner Springs, CA 92086",33.274217,-116.644207,NaN,None,"tuesday,efap",3rd Tuesday of the month from 8:00 am - 10:00 am,No rescheduled for 2026,Food menu changes monthly. Typical food packag...,Emergency Food Assistance Program Income Guide...,Drive-thru services only; Food Bank ID cards a...,"[tuesday, efap]"
154,Waterfront Park,"1600 Pacific Hwy, San Diego, CA 92101, US",32.721990,-117.172050,NaN,None,"wednesday,senior",3rd Wednesday of each month from 9:00 am - 10:...,No holiday reschedules,A 30 pound box of food every month that is fil...,Senior (60+) only; Senior Food Program Income ...,Walk-up services only. Please bring a cart & r...,"[wednesday, senior]"


In [9]:
DAY_TAGS = {"monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"}
SERVICE_TAGS = {"efap", "food to nonprofit", "neighborhood distribution", "senior"}
DISPLAY_NAMES = {
    "efap": "EFAP",
    "food to nonprofit": "Food To Nonprofit",
    "neighborhood distribution": "Neighborhood Distribution",
    "senior": "Senior",
}
 
 
def split_day(tags_list):
    days = [t for t in tags_list if t.lower() in DAY_TAGS]
    return ", ".join(d.title() for d in days) if days else None
 
 
def split_service_type(tags_list):
    services = [t for t in tags_list if t.lower() in SERVICE_TAGS]
    return ", ".join(DISPLAY_NAMES[s.lower()] for s in services) if services else None

In [10]:
def categorize_food(text):
    if not isinstance(text, str) or not text.strip():
        return None
    t = text.lower()
 
    has_tefap = "shelf-stable" in t and "menu changes" in t
    has_senior_box = "30 pound box" in t
    has_reduced_box = "15-25 pound box" in t
    has_produce = "pounds of fresh produce" in t or "produce; occasional dry" in t
    has_generic_dry = "nonperishable dry goods and fresh produce" in t
    is_data_error = "rescheduled for" in t and "shelf-stable" not in t and "pound box" not in t
 
    if is_data_error:
        return "DATA_ERROR_NOT_FOOD_DESCRIPTION"
 
    categories = []
    if has_tefap:
        categories.append("EFAP Standard Package")
    if has_senior_box:
        categories.append("Senior 30lb Box")
    if has_reduced_box:
        categories.append("Reduced 15-25lb Box")
    if has_produce:
        categories.append("Neighborhood Produce (25-30lb)")
    if has_generic_dry:
        categories.append("Nonperishable Dry Goods + Produce")
 
    return ", ".join(categories) if categories else "Other/Uncategorized"

In [11]:
def clean_eligibility(text):
    if not isinstance(text, str) or not text.strip():
        return None
    tl = text.strip().lower()
 
    has_efap = "emergency food assistance" in tl
    has_senior = "senior" in tl and "60+" in tl
    is_open = "all are welcome" in tl
 
    if has_efap and has_senior:
        return "EFAP + Senior (Income Guidelines Apply)"
    if has_efap:
        return "EFAP Income Guidelines Apply"
    if has_senior:
        return "Senior (60+) Income Guidelines Apply"
    if is_open:
        return "Open to All"
    return text.strip()
 
 
SERVICE_NOTE_FLAGS = {
    "walkup": ["walk-up", "walk up", "walk-in", "walk in"],
    "drivethru": ["drive-thru", "drive thru"],
    "id_card_available": ["id card"],
    "bring_cart": ["bring a cart", "bring cart"],
    "diapers_period_supplies": ["diaper", "period supp"],
    "appointment_required": ["appointment"],
    "line_number_system": ["line number"],
    "military_access_required": ["military base access"],
}
 
 
def clean_service_notes(text):
    if not isinstance(text, str) or not text.strip():
        return {flag: False for flag in SERVICE_NOTE_FLAGS}
    t = text.lower()
    return {
        flag: any(phrase in t for phrase in phrases)
        for flag, phrases in SERVICE_NOTE_FLAGS.items()
    }
 
 
MONTH_PATTERN = re.compile(
    r"\b(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\b", re.IGNORECASE
)
 
 
def clean_closures(text):
    if not isinstance(text, str) or not text.strip():
        return {"reschedule_status": "Unknown", "has_specific_dates": False}
    tl = text.strip().lower()
 
    if tl.startswith("no ") or "regularly scheduled days" in tl:
        return {"reschedule_status": "No Reschedules", "has_specific_dates": False}
 
    has_date = bool(MONTH_PATTERN.search(tl)) or bool(re.search(r"\d{1,2}/\d{1,2}", tl))
    if has_date:
        return {"reschedule_status": "Has Specific Reschedule Dates", "has_specific_dates": True}
 
    return {"reschedule_status": "Other", "has_specific_dates": False}
 
 
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
DAY_ALT = "|".join(DAYS)
 
MONTHLY_PATTERN = re.compile(
    rf"(?:\d(?:st|nd|rd|th)|last)(?:\s*(?:,|and|&)\s*(?:\d(?:st|nd|rd|th)|last))*"
    rf"\s+(?P<day>{DAY_ALT})",
    re.IGNORECASE,
)
WEEKLY_PATTERN = re.compile(
    rf"\b(?P<days>(?:{DAY_ALT})s?(?:\s*(?:,|&|and|-|to)\s*(?:{DAY_ALT})s?)*)\b",
    re.IGNORECASE,
)
TIME_RANGE_PATTERN = re.compile(
    r"(?P<start>\d{1,2}(?::\d{2})?\s*[ap]\.?m\.?)\s*(?:-|until|to)\s*"
    r"(?P<end>\d{1,2}(?::\d{2})?\s*[ap]\.?m\.?|food (?:is gone|runs out))",
    re.IGNORECASE,
)
SINGLE_TIME_PATTERN = re.compile(r"\d{1,2}(?::\d{2})?\s*[ap]\.?m\.?", re.IGNORECASE)
 
 
def clean_schedule(text):
    if not isinstance(text, str) or not text.strip():
        return {"frequency": None, "days_of_week": None, "start_time": None,
                "end_time": None, "schedule_needs_review": True}
 
    t = text.strip()
    m = MONTHLY_PATTERN.search(t)
 
    if m:
        frequency = "Monthly"
        days_of_week = m.group("day").title()
    else:
        dm = WEEKLY_PATTERN.search(t)
        frequency = "Weekly" if dm else None
        days_of_week = dm.group("days") if dm else None
 
    times = TIME_RANGE_PATTERN.search(t)
    if times:
        start_time, end_time = times.group("start"), times.group("end")
    else:
        singles = SINGLE_TIME_PATTERN.findall(t)
        start_time = singles[0] if singles else None
        end_time = singles[1] if len(singles) > 1 else None
 
    needs_review = days_of_week is None or start_time is None
 
    return {
        "frequency": frequency,
        "days_of_week": days_of_week,
        "start_time": start_time,
        "end_time": end_time,
        "schedule_needs_review": needs_review,
    }


In [12]:
def build_dataset(md_path):
    payload = load_from_file(md_path)
    df = to_dataframe(payload)
    df = df.drop(columns=["id"])
 
    df["zip"] = df["address"].apply(extract_zip)
    df["city"] = df["address"].apply(extract_city)
    df["county"] = df["city"].apply(get_county)
 
    df["is_church"] = df["name"].str.lower().str.contains("church", na=False)
    df["is_YMCA"] = df["name"].str.lower().str.contains("ymca", na=False)
 
    df["day"] = df["tags_list"].apply(split_day)
    df["service_type"] = df["tags_list"].apply(split_service_type)
    df = df.drop(columns=["tags_list", "tags"])
 
    df["No_Phone_Number"] = df["phone"].isna()
    df["No_Website"] = df["website"].isna()
 
    df["food_category"] = df["food_provided"].apply(categorize_food)
    df = df.drop(columns=["food_provided"])
 
    # cleaning step runs directly on this same in-memory df -- no disk
    # round-trip, so there's no way for it to silently clean stale data
    df["eligibility_clean"] = df["eligibility"].apply(clean_eligibility)
 
    service_flags = df["service_notes"].apply(clean_service_notes).apply(pd.Series)
    df = pd.concat([df, service_flags], axis=1)
 
    closure_result = df["closures_reschedules"].apply(clean_closures).apply(pd.Series)
    df = pd.concat([df, closure_result], axis=1)
 
    schedule_result = df["schedule"].apply(clean_schedule).apply(pd.Series)
    df = pd.concat([df, schedule_result], axis=1)
 
    return df
 
 
if __name__ == "__main__":
    df = build_dataset("san-diego-food-bank-locations.md")

In [13]:
df

,name,address,latitude,longitude,phone,website,schedule,closures_reschedules,eligibility,service_notes,...,appointment_required,line_number_system,military_access_required,reschedule_status,has_specific_dates,frequency,days_of_week,start_time,end_time,schedule_needs_review
0,Aguilas del Poderoso Dios,"5901 Rancho Hills Drive, San Diego, CA 92139",32.672252,-117.061430,NaN,None,1st and 3rd Thursday of the month 10:00am unti...,"Reschedules: January 8 and January 22, 2026. C...",All are welcome,"Walk-up & drive-thru services; if walking up, ...",...,False,False,False,Has Specific Reschedule Dates,True,Monthly,Thursday,10:00am,food runs out,False
1,All Saint Episcopal Church,"651 Eucalyptus Avenue, Vista, CA 92084",33.202163,-117.234913,NaN,None,3rd and 4th Saturday of each month; 11 AM - 12...,NaN,All are welcome,NaN,...,False,False,False,Unknown,False,Monthly,Saturday,11 AM,12 PM.,False
2,Apostolic Assembly Church,"1717 East Lincoln Avenue, Escondido, CA 92027",33.143996,-117.063386,NaN,None,3rd Saturday of the month from 9:30 am to 10:3...,"January 10, 2026",NaN,Drive-thru services only; Food Bank ID cards a...,...,False,False,False,Has Specific Reschedule Dates,True,Monthly,Saturday,9:30 am,10:30 am,False
3,Apostolic Assembly First Church San Diego,"611 South 35th St, San Diego, CA 92113",32.699690,-117.118156,NaN,None,2nd Thursday of each month from 10:00 am - 12:...,No holiday reschedules for 2026,Emergency Food Assistance Program Income Guide...,"Walk-up & drive-thru services; if walking up, ...",...,False,False,False,No Reschedules,False,Monthly,Thursday,10:00 am,12:00 pm,False
4,Armed Services YMCA Camp Pendleton,Building 200090 Ash Road and Wire Mountain Roa...,33.222690,-117.380070,NaN,None,4th Friday of the month from 9:15 am - 11:45 am,"Rescheduled to: May 29, Nov 20, Dec 11",Emergency Food Assistance Program Income Guide...,Military base access required to reach this site.,...,False,False,True,Has Specific Reschedule Dates,True,Monthly,Friday,9:15 am,11:45 am,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151,Us 4 Warriors,"900 Paseo del Rey, Chula Vista, CA 91910",32.633457,-117.027515,NaN,None,4th Tuesday of the month from 12:00pm - 1:00pm,All distributions will be held on regularly sc...,All are welcome,"Walk-up & drive-thru services; if walking up, ...",...,False,False,False,No Reschedules,False,Monthly,Tuesday,12:00pm,1:00pm,False
152,Victory Resource Center,"1220 Third Avenue, Suite 1 Chula Vista, CA 91911",32.609358,-117.067768,NaN,None,"Starting January 28th; Mondays, Wednesdays, an...","Closed in 2026 on: Feb 13, Feb 16, Mar 30, Apr...",Emergency Food Assistance Program Income Guide...,NaN,...,False,False,False,Has Specific Reschedule Dates,True,Weekly,"Mondays, Wednesdays",10:00 am,1:00 pm,False
153,Warner Springs Community Resource Center,"30950 Highway 79, Warner Springs, CA 92086",33.274217,-116.644207,NaN,None,3rd Tuesday of the month from 8:00 am - 10:00 am,No rescheduled for 2026,Emergency Food Assistance Program Income Guide...,Drive-thru services only; Food Bank ID cards a...,...,False,False,False,No Reschedules,False,Monthly,Tuesday,8:00 am,10:00 am,False
154,Waterfront Park,"1600 Pacific Hwy, San Diego, CA 92101, US",32.721990,-117.172050,NaN,None,3rd Wednesday of each month from 9:00 am - 10:...,No holiday reschedules,Senior (60+) only; Senior Food Program Income ...,Walk-up services only. Please bring a cart & r...,...,False,False,False,No Reschedules,False,Monthly,Wednesday,9:00 am,10:30 am,False


In [15]:
df = df.drop(columns=["schedule", "closures_reschedules", "eligibility", "service_notes", "phone", "website"])

In [16]:
df.to_csv("google_maps_cleaned_data.csv", index=False)